In [ ]:
!pip install -q datasets transformers evaluate rouge_score sacrebleu accelerate sentence-transformers gradio scikit-learn pandas


In [ ]:
from datasets import load_dataset
import numpy as np
import pandas as pd
import re

raw = load_dataset("truehealth/medicationqa")["train"]

def clean_text(x):
    if x is None:
        return ""
    x = str(x).replace("\n", " ").replace("\t", " ")
    x = re.sub(r"\s+", " ", x).strip()
    return x

def normalize_example(ex):
    return {
        "Question": clean_text(ex["Question"]),
        "Focus (Drug)": clean_text(ex["Focus (Drug)"]),
        "Question Type": clean_text(ex["Question Type"]),
        "Section Title": clean_text(ex["Section Title"]),
        "Answer": clean_text(ex["Answer"]),
        "URL": clean_text(ex["URL"]),
    }

data = raw.map(normalize_example)
data = data.filter(lambda x: len(x["Question"]) > 0 and len(x["Answer"]) > 0)

split1 = data.train_test_split(test_size=0.2, seed=42)
train_data = split1["train"]
temp_data = split1["test"]
split2 = temp_data.train_test_split(test_size=0.5, seed=42)
val_data = split2["train"]
test_data = split2["test"]

print(len(train_data), len(val_data), len(test_data))


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Using the latest cached version of the dataset since truehealth/medicationqa couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'default' at /root/.cache/huggingface/datasets/truehealth___medicationqa/default/0.0.0/f8ae96155ac234a54aaf2b70ca4e02d22cfe31af (last modified on Mon Apr 27 15:32:55 2026).


551 69 69


In [ ]:
def make_input(ex):
    return (
        "Answer the medication question using only reliable drug information. "
        "If interaction or safety is asked, mention that the user should ask a doctor/pharmacist.\n"
        f"Question: {ex['Question']}\n"
        f"Drug: {ex['Focus (Drug)']}\n"
        f"Question type: {ex['Question Type']}\n"
        f"Relevant section: {ex['Section Title']}\n"
        "Answer:"
    )

print(make_input(train_data[0]))
print("REFERENCE:", train_data[0]["Answer"][:300])


Answer the medication question using only reliable drug information. If interaction or safety is asked, mention that the user should ask a doctor/pharmacist.
Question: clonazepam ".25mg" lowest dosage?
Drug: clonazepam
Question type: Dose
Relevant section: HOW SUPPLIED
Answer:
REFERENCE: Klonopin Wafers (clonazepam orally disintegrating tablets) are white, round and debossed with the tablet strength … 0.125 mg debossed 1/8 …


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer

model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

max_input_length = 256
max_target_length = 128

def safe_text(x):
    if x is None:
        return ""
    return str(x).strip()

def tokenize_function(examples):
    inputs = []
    targets = []

    for i in range(len(examples["Question"])):
        ex = {k: examples[k][i] for k in examples.keys()}

        question = safe_text(ex.get("Question", ""))
        answer = safe_text(ex.get("Answer", ""))

        if question == "" or answer == "":
            continue

        inputs.append(make_input(ex))
        targets.append(answer)

    model_inputs = tokenizer(
        inputs,
        max_length=max_input_length,
        truncation=True,
        padding=False
    )

    labels = tokenizer(
        text_target=targets,
        max_length=max_target_length,
        truncation=True,
        padding=False
    )

    model_inputs["labels"] = labels["input_ids"]

    return model_inputs

tokenized_train = train_data.map(
    tokenize_function,
    batched=True,
    remove_columns=train_data.column_names
)

tokenized_val = val_data.map(
    tokenize_function,
    batched=True,
    remove_columns=val_data.column_names
)

tokenized_test = test_data.map(
    tokenize_function,
    batched=True,
    remove_columns=test_data.column_names
)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    label_pad_token_id=-100
)

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Map:   0%|          | 0/551 [00:00<?, ? examples/s]

Map:   0%|          | 0/69 [00:00<?, ? examples/s]

Map:   0%|          | 0/69 [00:00<?, ? examples/s]

In [ ]:
print("Train size:", len(tokenized_train))
print("Val size:", len(tokenized_val))
print("Test size:", len(tokenized_test))

print("Input:")
print(tokenizer.decode(tokenized_train[0]["input_ids"]))

print("\nLabel:")
print(tokenizer.decode(tokenized_train[0]["labels"]))

Train size: 551
Val size: 69
Test size: 69
Input:
Answer the medication question using only reliable drug information. If interaction or safety is asked, mention that the user should ask a doctor/pharmacist. Question: clonazepam ".25mg" lowest dosage? Drug: clonazepam Question type: Dose Relevant section: HOW SUPPLIED Answer:</s>

Label:
Klonopin Wafers (clonazepam orally disintegrating tablets) are white, round and debossed with the tablet strength <unk> 0.125 mg debossed 1/8 <unk></s>


In [ ]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer
training_args = Seq2SeqTrainingArguments(
    output_dir="./flan-t5-medicationqa",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=5,
    weight_decay=0.01,
    predict_with_generate=True,
    generation_max_length=128,
    generation_num_beams=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    save_total_limit=2,
    logging_steps=20,
    report_to="none",
    fp16=False,   # önemli
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    processing_class=tokenizer,
    data_collator=data_collator,
)

trainer.train()
trainer.save_model("./flan-t5-medicationqa-final")
tokenizer.save_pretrained("./flan-t5-medicationqa-final")

Epoch,Training Loss,Validation Loss
1,36.418198,8.542356
2,35.005124,8.099935
3,33.950790,7.928729
4,33.495609,7.861564
5,33.364404,7.799691


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight'].


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./flan-t5-medicationqa-final/tokenizer_config.json',
 './flan-t5-medicationqa-final/tokenizer.json')

In [ ]:
def ask_flan_t5(question, drug="", qtype="", section=""):
    ex = {
        "Question": clean_text(question),
        "Focus (Drug)": clean_text(drug),
        "Question Type": clean_text(qtype),
        "Section Title": clean_text(section),
    }
    input_text = make_input(ex)
    inputs = tokenizer(input_text, return_tensors="pt", max_length=max_input_length, truncation=True).to(model.device)

    with torch.no_grad():
        # outputs = model.generate(
        #     **inputs,
        #     max_new_tokens=160,
        #     min_new_tokens=20,
        #     num_beams=4,
        #     no_repeat_ngram_size=4,
        #     repetition_penalty=1.25,
        #     length_penalty=0.8,
        #     early_stopping=True,
        # )
        outputs = model.generate(
              **inputs,
              max_new_tokens=220,
              min_new_tokens=40,
              num_beams=4,
              no_repeat_ngram_size=4,
              repetition_penalty=1.2,
              length_penalty=1.1,
              early_stopping=True,
          )
    return tokenizer.decode(outputs[0], skip_special_tokens=True).strip()

ask_flan_t5(
    "how does rivastigmine and otc sleep medicine interact",
    drug="rivastigmine",
    qtype="Interaction",
    section="What special precautions should I follow?"
)


'The drug and the otc is used to make or take some of it in your body. If you are taking this, there may be an increase with other drugs that can help as well because they do not work for me so when we have more on them (and also how by)'

In [ ]:
rouge = evaluate.load("rouge")
bleu = evaluate.load("sacrebleu")

def f1_score(pred, ref):
    pred_tokens = pred.split()
    ref_tokens = ref.split()

    common = Counter(pred_tokens) & Counter(ref_tokens)
    num_same = sum(common.values())

    if num_same == 0:
        return 0

    precision = num_same / len(pred_tokens)
    recall = num_same / len(ref_tokens)

    return 2 * precision * recall / (precision + recall)

In [ ]:
def compute_metrics():
    predictions = []
    references = []

    sample_data = test_data.select(range(min(100, len(test_data))))

    for example in sample_data:
        question = example["Question"]
        true_answer = example["Answer"]

        pred = ask_flan_t5(
            question,
            drug=example.get("Focus (Drug)", ""),
            qtype=example.get("Question Type", ""),
            section=example.get("Section Title", "")
        )

        if pred is None or pred.strip() == "":
            pred = " "

        predictions.append(str(pred))
        references.append(str(true_answer))

    rouge_result = rouge.compute(predictions=predictions, references=references)

    bleu_result = bleu.compute(
        predictions=predictions,
        references=[[ref] for ref in references]
    )

    exact_match = np.mean([
        p.strip().lower() == r.strip().lower()
        for p, r in zip(predictions, references)
    ])

    f1 = np.mean([f1_score(p, r) for p, r in zip(predictions, references)])

    return rouge_result, bleu_result, exact_match, f1

In [ ]:
rouge_result, bleu_result, exact_match, f1 = compute_metrics()

print("ROUGE:", rouge_result)
print("BLEU:", bleu_result)
print("Exact Match:", exact_match)
print("F1 Score:", f1)

ROUGE: {'rouge1': np.float64(0.16311814524174972), 'rouge2': np.float64(0.011212409421570665), 'rougeL': np.float64(0.09126099360566664), 'rougeLsum': np.float64(0.09119360650336872)}
BLEU: {'score': 0.38841932038530996, 'counts': [758, 53, 5, 0], 'totals': [3806, 3737, 3668, 3599], 'precisions': [19.915922228060957, 1.4182499331014182, 0.13631406761177753, 0.013892747985551542], 'bp': 0.8076632401489808, 'sys_len': 3806, 'ref_len': 4619}
Exact Match: 0.0
F1 Score: 0.13877426291202746


In [ ]:
import shutil

shutil.make_archive("flan_t5_model", 'zip', "flan-t5-medicationqa-final")

'/content/flan_t5_model.zip'

In [ ]:
from google.colab import files
files.download("flan_t5_model.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>